In [1]:
import numpy as np
import pandas as pd

from SymbolicDSGE import DSGESolver, ModelParser, Shock
from SymbolicDSGE.monte_carlo import MCPipeline
from SymbolicDSGE.monte_carlo.operations.core import (
    reference_filter_step,
    simulation_step,
)
from SymbolicDSGE.monte_carlo.operations.regressions import regression_step
from SymbolicDSGE.monte_carlo.operations.tests import (
    ljung_box_test_step,
    wald_test_step,
)
import cProfile

In [2]:
model, kalman = ModelParser("../../MODELS/misspec_test/reference.yaml").get_all()

solver = DSGESolver(model, kalman)
compiled = solver.compile(
    n_state=3,
    n_exog=3,
)

steady_state = np.zeros(5, dtype=np.float64)
reference = solver.solve(compiled, steady_state=steady_state)

dgp_model, dgp_kalman = ModelParser("../../MODELS/misspec_test/misspec.yaml").get_all()
dgp_comp = DSGESolver(dgp_model, dgp_kalman).compile(n_state=3, n_exog=3)
dgp_sol = DSGESolver(dgp_model, dgp_kalman)
dgp = dgp_sol.solve(dgp_comp, steady_state=steady_state)

In [3]:
T = 200
n_obs = len(reference.compiled.observable_names)

pipeline = MCPipeline(
    [
        simulation_step(
            T=T,
            shocks={
                "g,z": Shock(T=T, dist="norm", multivar=True, seed=0),
                "r": Shock(T=T, dist="norm", seed=1),
            },
            observables=True,
        ),
        reference_filter_step(estimate_R_diag=False),
        wald_test_step(
            "std_innov_mean",
            source="std_innov",
            target=np.zeros(n_obs),
            kind="mean",
            burn_in=20,
        ),
        ljung_box_test_step(
            "OGap_autocorr",
            source="std_innov",
            column=0,
            lags=10,
            burn_in=20,
        ),
        ljung_box_test_step(
            "Infl_autocorr",
            source="std_innov",
            column=1,
            lags=10,
            burn_in=20,
        ),
        ljung_box_test_step(
            "Rate_autocorr",
            source="std_innov",
            column=2,
            lags=10,
            burn_in=20,
        ),
        wald_test_step(
            "std_innov_second_moment",
            source="std_innov",
            target=np.eye(n_obs),
            kind="second_moment",
            burn_in=20,
        ),
        regression_step(
            "OutGap_regression",
            kind="ridge_gs",
            start=1e-6,
            stop=1e2,
            num=10,
            X_source="x_pred",
            y_source="innov",
            y_column=0,
            X_columns=[2, 3, 4],
            variables=["r", "x", "Pi"],
        ),
    ]
)

In [7]:
mc = pipeline.run(
    reference=reference,
    dgp=dgp,
    n_rep=1000,
    retain_payloads=False,
    retain_test_results=False,
    verbosity=2,
)

datagen concluded successfully with 1346.91 it/s.
filter concluded successfully with 1718.43 it/s.
std_innov_mean concluded successfully with 15748.63 it/s.
OGap_autocorr concluded successfully with 37114.57 it/s.
Infl_autocorr concluded successfully with 53371.19 it/s.
Rate_autocorr concluded successfully with 60151.81 it/s.
std_innov_second_moment concluded successfully with 6308.80 it/s.
OutGap_regression concluded successfully with 5762.74 it/s.


In [5]:
t_summary = pd.DataFrame(
    {
        name: {
            "mean_statistic": res.mean_statistic,
            "mean_pval": res.mean_pval,
            "rejection_rate": res.rejection_rate,
            "ci_low": res.pval_confidence_interval()[0],
            "ci_high": res.pval_confidence_interval()[1],
        }
        for name, res in mc.test_summaries.items()
    }
).T
t_summary.round(3)

,mean_statistic,mean_pval,rejection_rate,ci_low,ci_high
std_innov_mean,2.531,0.569,0.035,0.025,0.048
OGap_autocorr,88.115,0.000,1.000,0.996,1.000
Infl_autocorr,125.037,0.000,1.000,0.996,1.000
Rate_autocorr,22.446,0.076,0.673,0.643,0.701
std_innov_second_moment,3000.276,0.000,1.000,0.996,1.000


In [6]:
print(
    any(
        res.value != 0
        for res in mc.regression_summaries["OutGap_regression"].status_trace
    )
)

(
    mc.regression_summaries["OutGap_regression"].coefficients.mean(axis=0).round(2),
    mc.regression_summaries["OutGap_regression"].r2_trace.mean().round(4),
)

False


(array([ 0.  , -0.19, -0.05, -3.79]), np.float64(0.1186))